In [1]:
import os
import json
import random
import shutil

# =========================
# CONFIG
# =========================

TRAIN_IMAGES = "Dataset/train/"
TEST_IMAGES = "Dataset/test/"

TRAIN_JSON = "Dataset/train/_annotations.coco.json"
TEST_JSON = "Dataset/test/_annotations.coco.json"

TARGET_CLASS_NAME = "Clothes"
NUM_IMAGES = 10

# =========================
# LOAD JSON FILES
# =========================

with open(TRAIN_JSON, "r") as f:
    train_data = json.load(f)

with open(TEST_JSON, "r") as f:
    test_data = json.load(f)

# =========================
# FIND CATEGORY ID
# =========================

category_id = None

for cat in train_data["categories"]:
    if cat["name"] == TARGET_CLASS_NAME:
        category_id = cat["id"]
        break

if category_id is None:
    raise ValueError("Class not found!")

# =========================
# FIND IMAGES WITH CLOTHES
# =========================

image_ids_with_class = set()

for ann in train_data["annotations"]:
    if ann["category_id"] == category_id:
        image_ids_with_class.add(ann["image_id"])

# =========================
# SELECT IMAGES
# =========================

candidate_images = [
    img for img in train_data["images"]
    if img["id"] in image_ids_with_class
]

selected_images = random.sample(
    candidate_images,
    min(NUM_IMAGES, len(candidate_images))
)

selected_ids = set(img["id"] for img in selected_images)

print(f"Moving {len(selected_images)} images")

# =========================
# MOVE IMAGE FILES
# =========================

for img in selected_images:

    filename = img["file_name"]

    src = os.path.join(TRAIN_IMAGES, filename)
    dst = os.path.join(TEST_IMAGES, filename)

    if os.path.exists(src):
        shutil.move(src, dst)

# =========================
# MOVE IMAGE ENTRIES
# =========================

train_data["images"] = [
    img for img in train_data["images"]
    if img["id"] not in selected_ids
]

test_data["images"].extend(selected_images)

# =========================
# MOVE ANNOTATIONS
# =========================

moved_annotations = [
    ann for ann in train_data["annotations"]
    if ann["image_id"] in selected_ids
]

train_data["annotations"] = [
    ann for ann in train_data["annotations"]
    if ann["image_id"] not in selected_ids
]

test_data["annotations"].extend(moved_annotations)

# =========================
# SAVE UPDATED JSONS
# =========================

with open(TRAIN_JSON, "w") as f:
    json.dump(train_data, f)

with open(TEST_JSON, "w") as f:
    json.dump(test_data, f)

print("Done!")

Moving 10 images
Done!


In [ ]:
# import os
# import random
# import shutil

# # =========================
# # CONFIG
# # =========================

# TRAIN_IMAGES = "Dataset_YOLO/train/images"
# TEST_IMAGES = "Dataset_YOLO/test/images"

# TRAIN_LABELS = "Dataset_YOLO/train/labels"
# TEST_LABELS = "Dataset_YOLO/test/labels"

# TARGET_CLASS_ID = 1
# NUM_IMAGES = 10

# # =========================
# # FIND IMAGES CONTAINING CLASS
# # =========================

# candidate_files = []

# for label_file in os.listdir(TRAIN_LABELS):

#     if not label_file.endswith(".txt"):
#         continue

#     label_path = os.path.join(TRAIN_LABELS, label_file)

#     with open(label_path, "r") as f:
#         lines = f.readlines()

#     found = False

#     for line in lines:

#         class_id = int(line.split()[0])

#         if class_id == TARGET_CLASS_ID:
#             found = True
#             break

#     if found:
#         candidate_files.append(label_file)

# # =========================
# # SELECT FILES
# # =========================

# selected_files = random.sample(
#     candidate_files,
#     min(NUM_IMAGES, len(candidate_files))
# )

# print(f"Moving {len(selected_files)} files")

# # =========================
# # MOVE FILES
# # =========================

# for label_file in selected_files:

#     base_name = os.path.splitext(label_file)[0]

#     # Find image extension
#     image_file = None

#     for ext in [".jpg", ".jpeg", ".png"]:

#         temp = base_name + ext

#         if os.path.exists(os.path.join(TRAIN_IMAGES, temp)):
#             image_file = temp
#             break

#     if image_file is None:
#         continue

#     # Paths
#     src_img = os.path.join(TRAIN_IMAGES, image_file)
#     dst_img = os.path.join(TEST_IMAGES, image_file)

#     src_lbl = os.path.join(TRAIN_LABELS, label_file)
#     dst_lbl = os.path.join(TEST_LABELS, label_file)

#     # Move
#     shutil.move(src_img, dst_img)
#     shutil.move(src_lbl, dst_lbl)

# print("Done!")

Moving 10 files
Done!
